# PRAM Lab 2 — Amdahl vs Gustafson
The **model calculations are in C++**. Python is used only to display the curves and interactive controls.

The lab answers two different questions:

- **Strong scaling / Amdahl:** fixed problem size. How much faster can the same problem become?
- **Scaled problem / Gustafson:** grow the parallel part with `P`. How much more work can fit in the same time budget?

Use the serial-fraction slider and watch the two curves react immediately.

In [1]:
# Python setup: controls + plotting only.
import importlib.util, json, subprocess, sys
if importlib.util.find_spec('ipywidgets') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','ipywidgets'])
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

In [2]:
%%writefile /tmp/amdahl_gustafson.cpp
#include <cmath>
#include <cstdlib>
#include <iostream>
#include <iomanip>
#include <string>
using namespace std;

double amdahl(double P,double s){ return 1.0/(s+(1.0-s)/P); }
double gustafson(double P,double s){ return s+(1.0-s)*P; }

int main(int argc,char** argv){
    if(argc<4){ cerr << "usage: scaling s maxP step\n"; return 2; }
    double s=stod(argv[1]); int maxP=stoi(argv[2]); int step=stoi(argv[3]);
    cout<<setprecision(12);
    cout << "{\"serial_fraction\":"<<s<<",\"rows\":[";
    bool first=true;
    for(int P=1;P<=maxP;P+=step){
        if(!first) cout<<','; first=false;
        double a=amdahl(P,s), g=gustafson(P,s);
        cout << "{\"P\":"<<P
             <<",\"amdahl\":"<<a
             <<",\"gustafson\":"<<g
             <<",\"amdahl_efficiency\":"<<(a/P)
             <<",\"gustafson_efficiency\":"<<(g/P)<<"}";
    }
    cout << "]}";
}

Writing /tmp/amdahl_gustafson.cpp


In [3]:
subprocess.run(['g++','-std=c++17','-O2','/tmp/amdahl_gustafson.cpp','-o','/tmp/amdahl_gustafson'],check=True)
print('C++ model compiled.')

def scaling_cpp(s=0.05,maxP=256,step=1):
    raw=subprocess.check_output(['/tmp/amdahl_gustafson',str(float(s)),str(int(maxP)),str(int(step))],text=True)
    return json.loads(raw)

# Sanity checks are against the analytical definitions, but the plotted data come from C++.
t=scaling_cpp(.05,8,1)['rows']
assert abs(t[0]['amdahl']-1.0)<1e-12
assert abs(t[-1]['amdahl'] - 1/(.05+.95/8)) < 1e-10
assert abs(t[-1]['gustafson'] - (.05+.95*8)) < 1e-10
print('Sanity checks passed.')

C++ model compiled.
Sanity checks passed.


## Interactive experiment

**Actionable reading:** move the serial fraction from 1% to 20%. The orange marker shows the first `P` where Amdahl efficiency falls below the selected efficiency threshold. That is a practical “stop adding processors” point for a fixed-size experiment.

In [4]:
def draw_scaling(s=.05,maxP=256,eff_threshold=.60):
    d=scaling_cpp(s,maxP,1)
    df=pd.DataFrame(d['rows'])
    fig,axs=plt.subplots(1,2,figsize=(14,4.8))
    axs[0].plot(df.P,df.amdahl,label='Amdahl: fixed size',lw=2.5)
    axs[0].plot(df.P,df.gustafson,label='Gustafson: scaled problem',lw=2.5)
    axs[0].plot(df.P,df.P,'--',lw=1,label='ideal P')
    axs[0].set_xlabel('processors P'); axs[0].set_ylabel('speedup')
    axs[0].set_title(f'C++ scaling model — serial fraction s={s:.1%}')
    axs[0].grid(alpha=.25); axs[0].legend()

    axs[1].plot(df.P,df.amdahl_efficiency,label='Amdahl efficiency',lw=2.5)
    axs[1].axhline(eff_threshold,ls='--',label=f'threshold {eff_threshold:.0%}')
    below=df[df.amdahl_efficiency<eff_threshold]
    stop=int(below.iloc[0].P) if len(below) else None
    if stop:
        row=df[df.P==stop].iloc[0]
        axs[1].scatter([stop],[row.amdahl_efficiency],s=90,zorder=5)
        axs[1].annotate(f'first below threshold: P={stop}',(stop,row.amdahl_efficiency),xytext=(10,18),textcoords='offset points',arrowprops={'arrowstyle':'->'})
    axs[1].set_ylim(0,1.05); axs[1].set_xlabel('processors P'); axs[1].set_ylabel('efficiency')
    axs[1].set_title('Fixed-size decision view')
    axs[1].grid(alpha=.25); axs[1].legend()
    plt.show()

    p_show=maxP
    row=df.iloc[-1]
    print(f'At P={p_show}: Amdahl speedup={row.amdahl:.2f}, efficiency={row.amdahl_efficiency:.1%}; Gustafson scaled speedup={row.gustafson:.2f}')

s_w=widgets.FloatSlider(value=.05,min=.0,max=.30,step=.01,description='serial s:',readout_format='.0%',continuous_update=True)
p_w=widgets.Dropdown(options=[32,64,128,256,512,1024],value=256,description='max P:')
e_w=widgets.FloatSlider(value=.60,min=.20,max=.90,step=.05,description='min E:',readout_format='.0%',continuous_update=True)
out=widgets.Output()
def rerender(*_):
    with out:
        clear_output(wait=True); draw_scaling(s_w.value,p_w.value,e_w.value)
for w in (s_w,p_w,e_w): w.observe(rerender,names='value')
display(widgets.VBox([widgets.HBox([s_w,p_w,e_w]),out])); rerender()

### Try this

1. Set `s = 1%`: note how many processors remain useful under strong scaling.
2. Set `s = 10%`: compare the strong-scaling ceiling with the scaled Gustafson curve.
3. Change the efficiency threshold to match the criterion you would use in a real benchmark.

The point is not that Gustafson “beats” Amdahl: they hold **different things fixed**.